In [ ]:
import os
import math
import random
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy.signal import butter, filtfilt, resample_poly, find_peaks
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

In [ ]:
# =========================================
# 1. Base path
# =========================================
BASE_PATH = "/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset/PPG_FieldStudy"

In [ ]:
# =========================================
# 2. Find all subject folders
# =========================================
subjects = sorted([
    folder for folder in os.listdir(BASE_PATH)
    if folder.startswith("S") and os.path.isdir(os.path.join(BASE_PATH, folder))
])

print("Subjects found:")
print(subjects)
print("Total subjects:", len(subjects))

In [ ]:
# =========================================
# 3. Load all subjects
# =========================================
all_data = {}

for subject in subjects:
    subject_path = os.path.join(BASE_PATH, subject)
    pkl_path = os.path.join(subject_path, f"{subject}.pkl")

    if not os.path.exists(pkl_path):
        print(f"Skipping {subject} - .pkl file not found")
        continue

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    subject_dict = {}

    # Extract wrist signals
    if "signal" in data and "wrist" in data["signal"]:
        wrist = data["signal"]["wrist"]
        subject_dict["bvp"] = np.array(wrist["BVP"]) if "BVP" in wrist else None
        subject_dict["acc_wrist"] = np.array(wrist["ACC"]) if "ACC" in wrist else None
        subject_dict["eda_wrist"] = np.array(wrist["EDA"]) if "EDA" in wrist else None
        subject_dict["temp_wrist"] = np.array(wrist["TEMP"]) if "TEMP" in wrist else None
    else:
        subject_dict["bvp"] = None
        subject_dict["acc_wrist"] = None
        subject_dict["eda_wrist"] = None
        subject_dict["temp_wrist"] = None

    # Extract chest signals
    if "signal" in data and "chest" in data["signal"]:
        chest = data["signal"]["chest"]
        subject_dict["ecg"] = np.array(chest["ECG"]) if "ECG" in chest else None
        subject_dict["acc_chest"] = np.array(chest["ACC"]) if "ACC" in chest else None
        subject_dict["resp"] = np.array(chest["Resp"]) if "Resp" in chest else None
        subject_dict["eda_chest"] = np.array(chest["EDA"]) if "EDA" in chest else None
        subject_dict["temp_chest"] = np.array(chest["Temp"]) if "Temp" in chest else None
        subject_dict["emg"] = np.array(chest["EMG"]) if "EMG" in chest else None
    else:
        subject_dict["ecg"] = None
        subject_dict["acc_chest"] = None
        subject_dict["resp"] = None
        subject_dict["eda_chest"] = None
        subject_dict["temp_chest"] = None
        subject_dict["emg"] = None

    # Labels / activity
    subject_dict["label"] = np.array(data["label"]) if "label" in data else None
    subject_dict["activity"] = np.array(data["activity"]) if "activity" in data else None

    all_data[subject] = subject_dict

print("\nFinished loading all subjects.")
print("Loaded subjects:", list(all_data.keys()))


In [ ]:

print("\n========== DATA SUMMARY ==========\n")

for subject, subject_data in all_data.items():
    print(f"{subject}:")
    
    for key, value in subject_data.items():
        if value is None:
            print(f"  {key}: None")
        else:
            print(f"  {key}: shape = {value.shape}, dtype = {value.dtype}")
    
    print("-" * 50)

In [ ]:
print("\n========== SAMPLE VALUES ==========\n")

for subject, subject_data in all_data.items():
    print(f"{subject}:")
    
    if subject_data["bvp"] is not None:
        print("  First 5 BVP values:", subject_data["bvp"][:5].flatten())
    
    if subject_data["ecg"] is not None:
        print("  First 5 ECG values:", subject_data["ecg"][:5].flatten())
    
    if subject_data["acc_wrist"] is not None:
        print("  First 3 wrist ACC values:")
        print(subject_data["acc_wrist"][:3])
    
    print("-" * 50)


In [ ]:
all_bvp_list = []
all_ecg_list = []

for subject, subject_data in all_data.items():
    if subject_data["bvp"] is not None:
        all_bvp_list.append(subject_data["bvp"].flatten())
    if subject_data["ecg"] is not None:
        all_ecg_list.append(subject_data["ecg"].flatten())

all_bvp = np.concatenate(all_bvp_list, axis=0)
all_ecg = np.concatenate(all_ecg_list, axis=0)

print("Combined BVP shape:", all_bvp.shape)
print("Combined ECG shape:", all_ecg.shape)

print("First 10 combined BVP values:", all_bvp[:10])
print("First 10 combined ECG values:", all_ecg[:10])


In [ ]:
subject_to_plot = "S1"

bvp = all_data[subject_to_plot]["bvp"]
ecg = all_data[subject_to_plot]["ecg"]

FS_BVP = 64
FS_ECG = 700

plot_seconds_bvp = 10
plot_seconds_ecg = 5

n_bvp = FS_BVP * plot_seconds_bvp
n_ecg = FS_ECG * plot_seconds_ecg

bvp_segment = bvp[:n_bvp].flatten()
ecg_segment = ecg[:n_ecg].flatten()

t_bvp = np.arange(len(bvp_segment)) / FS_BVP
t_ecg = np.arange(len(ecg_segment)) / FS_ECG

plt.figure(figsize=(14, 4))
plt.plot(t_bvp, bvp_segment)
plt.title(f"{subject_to_plot} - Wrist BVP (First {plot_seconds_bvp} sec)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(t_ecg, ecg_segment)
plt.title(f"{subject_to_plot} - Chest ECG (First {plot_seconds_ecg} sec)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 0. Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ============================================================
# 1. Assumptions from DALIA
# ============================================================
# Your Code.txt loads:
# all_data[subject]["bvp"]  -> wrist BVP (64 Hz)
# all_data[subject]["ecg"]  -> chest ECG (700 Hz)
#
# If your variable name is different, change here.
# ============================================================

assert "all_data" in globals(), "Please run your DALIA loading code first so that all_data exists."

subjects = sorted(all_data.keys())
print("Subjects:", subjects)
print("Number of subjects:", len(subjects))

In [ ]:
# ============================================================
# 2. Paper-faithful preprocessing parameters
# ============================================================
# Paper:
# - resample PPG to 128 Hz
# - Butterworth bandpass 0.5-8 Hz
# - 4 sec windows for training = 512 samples
# - 25% overlap
# - min-max normalize to [-1, 1]
# ============================================================

ORIG_FS_BVP = 64  # PPG samplig frequency(Original)
ORIG_FS_ECG = 700 # EEG sampling frequency (Original)
TARGET_FS = 128 # Target sampling frequency

TRAIN_WINDOW_SEC = 4 # Training window length
TRAIN_WINDOW_SAMPLES = TARGET_FS * TRAIN_WINDOW_SEC   # 512 = 4 x 128 samples per sec.

OVERLAP_RATIO = 0.25
STEP_SAMPLES = int(TRAIN_WINDOW_SAMPLES * (1 - OVERLAP_RATIO))  # 384

# ============================================================
# Longer windows for validation / test HR estimation
# ============================================================
EVAL_WINDOW_SEC = 12
EVAL_WINDOW_SAMPLES = EVAL_WINDOW_SEC * TARGET_FS
EVAL_STEP_SAMPLES = EVAL_WINDOW_SAMPLES // 2

BANDPASS_LOW = 0.5 #lower cutoff frequency 
BANDPASS_HIGH = 8.0 #upper cutoff frequency
BUTTER_ORDER = 4 #butterworth filter order

BATCH_SIZE = 64 #Batch sizes
EPOCHS = 12

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_GEN_PATH = os.path.join(CHECKPOINT_DIR, "best_generator.weights.h5")
LAST_GEN_PATH = os.path.join(CHECKPOINT_DIR, "last_generator.weights.h5")

BEST_DISC_PATH = os.path.join(CHECKPOINT_DIR, "best_discriminator.weights.h5")
LAST_DISC_PATH = os.path.join(CHECKPOINT_DIR, "last_discriminator.weights.h5")

In [ ]:
# ============================================================
# 3. Signal utilities
# ============================================================

#butter worthe filter
def butter_bandpass_filter(signal, fs, lowcut=0.5, highcut=8.0, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


#min_max Normalization
def minmax_scale_minus1_to_1(x):
    x = np.asarray(x, dtype=np.float32)
    xmin = np.min(x)
    xmax = np.max(x)
    if np.isclose(xmax, xmin):
        return np.zeros_like(x, dtype=np.float32)
    return 2.0 * (x - xmin) / (xmax - xmin) - 1.0

# Signal Segmentation
def segment_signal(signal, window_size, step_size):
    segments = []
    for start in range(0, len(signal) - window_size + 1, step_size):
        end = start + window_size
        segments.append(signal[start:end])
    return np.array(segments)


# convert in to 1D numpy array
def safe_flatten(x):
    return np.asarray(x).astype(np.float32).reshape(-1)

In [ ]:
#Estimate the hearate from ECG
def estimate_hr_from_ecg_window_old(ecg_window, fs=TARGET_FS):
    """
    Estimate HR from an ECG segment using R-peak detection.
    Returns bpm or np.nan if not enough reliable peaks.
    """
    x = np.asarray(ecg_window).astype(np.float32)

    # normalize
    x = x - np.mean(x)
    std = np.std(x) + 1e-8
    x = x / std

    # minimum distance between beats:
    # max HR ~ 220 bpm => min RR ~ 60/220 sec
    min_distance = int(fs * (60.0 / 220.0))
    if min_distance < 1:
        min_distance = 1

    # prominence threshold
    peaks, props = find_peaks(x, distance=min_distance, prominence=0.5)

    if len(peaks) < 2:
        return np.nan

    rr_intervals = np.diff(peaks) / fs  # seconds
    rr_intervals = rr_intervals[(rr_intervals > 0.27) & (rr_intervals < 2.0)]  # physiologic filter

    if len(rr_intervals) == 0:
        return np.nan

    hr = 60.0 / np.mean(rr_intervals)
    return hr
#Estimate heart rate with 700Hz ecg
def bandpass_ecg(ecg, fs=700, lowcut=5.0, highcut=25.0, order=2):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, ecg)

def estimate_hr_from_ecg_window_original_fs(ecg_window, fs=700):
    x = np.asarray(ecg_window).astype(np.float32)

    x = bandpass_ecg(x, fs=fs, lowcut=5.0, highcut=25.0, order=2)
    x = x - np.mean(x)
    x = x / (np.std(x) + 1e-8)

    min_distance = int(fs * 60.0 / 220.0)

    peaks, _ = find_peaks(
        x,
        distance=min_distance,
        prominence=0.8
    )

    if len(peaks) < 2:
        return np.nan

    rr_intervals = np.diff(peaks) / fs
    rr_intervals = rr_intervals[(rr_intervals > 0.27) & (rr_intervals < 2.0)]

    if len(rr_intervals) == 0:
        return np.nan

    hr = 60.0 / np.mean(rr_intervals)
    return hr

#Heart rate estimate with the output of the generater
def estimate_hr_from_generated_ppg_old(ppg_window, fs=TARGET_FS):
    """
    HR from generated/denoised PPG by systolic peak detection.
    """
    x = np.asarray(ppg_window).astype(np.float32)
    x = x - np.mean(x)

    # Physiological distance:
    # max HR ~ 220 bpm => min peak distance
    min_distance = int(fs * (60.0 / 220.0))
    if min_distance < 1:
        min_distance = 1

    prominence = max(0.05, 0.25 * np.std(x))
    peaks, _ = find_peaks(x, distance=min_distance, prominence=prominence)

    if len(peaks) < 2:
        return np.nan

    pp_intervals = np.diff(peaks) / fs
    pp_intervals = pp_intervals[(pp_intervals > 0.27) & (pp_intervals < 2.0)]

    if len(pp_intervals) == 0:
        return np.nan

    hr = 60.0 / np.mean(pp_intervals)
    return hr
def estimate_hr_from_generated_ppg_robust(ppg_window, fs=TARGET_FS):
    """
    More stable HR estimation from generated PPG.
    """
    x = np.asarray(ppg_window).astype(np.float32)

    x = butter_bandpass_filter(x, fs=fs, lowcut=0.7, highcut=4.0, order=3)

    x = x - np.mean(x)
    x = x / (np.std(x) + 1e-8)

    min_distance = int(fs * 60.0 / 200.0)

    peaks, _ = find_peaks(
        x,
        distance=min_distance,
        prominence=0.3,
        width=2
    )

    if len(peaks) < 2:
        return np.nan

    intervals = np.diff(peaks) / fs
    intervals = intervals[(intervals > 0.3) & (intervals < 1.5)]

    if len(intervals) == 0:
        return np.nan

    hr = 60.0 / np.median(intervals)
    return hr

In [ ]:
# ============================================================
# 5. Ideal PPG synthesis from the paper
# ============================================================
# Paper formula:
# PPG_i(t) = 0.8 cos(2π f_HR t + θ) - 0.35 sin(2(2π f_HR t + θ))
# ============================================================

def synthesize_ideal_ppg(hr_bpm, length=TRAIN_WINDOW_SAMPLES, fs=TARGET_FS, theta=None):
    if theta is None:
        theta = np.random.uniform(0, 2*np.pi)

    f_hr = hr_bpm / 60.0  # Hz
    t = np.arange(length) / fs

    ideal = 0.8 * np.cos(2 * np.pi * f_hr * t + theta) \
            - 0.35 * np.sin(2 * (2 * np.pi * f_hr * t + theta))

    ideal = minmax_scale_minus1_to_1(ideal)
    return ideal.astype(np.float32)

In [ ]:
x=synthesize_ideal_ppg(72,1000,128,90)
plt.plot(x)

In [ ]:
# ============================================================
# 6. Subject-wise preprocessing
# ============================================================

def preprocess_subject(subject_name, subject_dict):
    """
    Steps:
    - take BVP and ECG
    - resample both to 128 Hz
    - filter PPG with 0.5-8 Hz Butterworth
    - segment into 4-second windows with 25% overlap
    - normalize PPG windows to [-1,1]
    - estimate ECG HR per window
    - synthesize ideal PPG target using ECG HR
    """
    bvp = safe_flatten(subject_dict["bvp"])
    ecg = safe_flatten(subject_dict["ecg"])

    # Resample to 128 Hz
    # bvp: 64 -> 128
    bvp_rs = resample_poly(bvp, up=2, down=1) #since the origina BVP 64 we multiply it by 2
    #128 / 64 = 2 / 1

    # ecg: 700 -> 128
    # approximate rational resampling
    # keep ECG in original 700 Hz for HR estimation
    ecg_window_size_orig = int(TRAIN_WINDOW_SEC * ORIG_FS_ECG)
    ecg_step_orig = int(STEP_SAMPLES / TARGET_FS * ORIG_FS_ECG)
    
    ecg_windows_orig = segment_signal(ecg, ecg_window_size_orig, ecg_step_orig)

    # Align lengths

    # Band-pass PPG only 
    bvp_filt = butter_bandpass_filter(
        bvp_rs,
        fs=TARGET_FS,
        lowcut=BANDPASS_LOW,
        highcut=BANDPASS_HIGH,
        order=BUTTER_ORDER
    )

    # Segment
    bvp_windows = segment_signal(bvp_filt, TRAIN_WINDOW_SAMPLES, STEP_SAMPLES)
    

    X_noisy = []
    X_ideal = []
    y_hr = []
    meta = []

    for i in range(len(bvp_windows)):
        ppg_w = bvp_windows[i]
        ecg_w = ecg_windows[i]

        hr = estimate_hr_from_ecg_window_original_fs(ecg_w_orig, fs=ORIG_FS_ECG)
        if np.isnan(hr):
            continue

        # Keep only physiologic HR range from paper idea
        if hr < 40 or hr > 200:
            continue

        ppg_w = minmax_scale_minus1_to_1(ppg_w)
        ideal_w = synthesize_ideal_ppg(hr_bpm=hr, length=TRAIN_WINDOW_SAMPLES, fs=TARGET_FS)

        X_noisy.append(ppg_w.astype(np.float32))
        X_ideal.append(ideal_w.astype(np.float32))
        y_hr.append(np.float32(hr))
        meta.append((subject_name, i))

    if len(X_noisy) == 0:
        return None

    X_noisy = np.array(X_noisy, dtype=np.float32)[..., np.newaxis]
    X_ideal = np.array(X_ideal, dtype=np.float32)[..., np.newaxis]
    y_hr = np.array(y_hr, dtype=np.float32)

    return {
        "X_noisy": X_noisy,
        "X_ideal": X_ideal,
        "y_hr": y_hr,
        "meta": meta
    }


In [ ]:
# ============================================================
# 7. Build full dataset subject-wise
# ============================================================

subject_processed = {}

for s in tqdm(subjects, desc="Preprocessing subjects"):
    out = preprocess_subject(s, all_data[s])
    if out is not None:
        subject_processed[s] = out

valid_subjects = sorted(subject_processed.keys())
print("Usable subjects:", valid_subjects)
print("Usable subject count:", len(valid_subjects))

In [ ]:
# ============================================================
# 8. Subject-wise split
# ============================================================
# Paper uses participant-separated splits.
# Since you also want a best validation model, we do:
# - 80% train_val subjects
# - 20% test subjects
# - from train_val, carve out validation subjects
#
# This avoids leakage across windows of the same subject.
# ============================================================

trainval_subjects, test_subjects = train_test_split(
    valid_subjects, test_size=0.20, random_state=SEED
)

train_subjects, val_subjects = train_test_split(
    trainval_subjects, test_size=0.20, random_state=SEED
)

print("Train subjects:", train_subjects)
print("Val subjects:", val_subjects)
print("Test subjects:", test_subjects)

def gather_subjects(subject_list):
    Xn, Xi, y, meta = [], [], [], []
    for s in subject_list:
        Xn.append(subject_processed[s]["X_noisy"])
        Xi.append(subject_processed[s]["X_ideal"])
        y.append(subject_processed[s]["y_hr"])
        meta.extend(subject_processed[s]["meta"])
    return (
        np.concatenate(Xn, axis=0),
        np.concatenate(Xi, axis=0),
        np.concatenate(y, axis=0),
        meta
    )

X_train_noisy, X_train_ideal, y_train_hr, meta_train = gather_subjects(train_subjects)
X_val_noisy, X_val_ideal, y_val_hr, meta_val = gather_subjects(val_subjects)
X_test_noisy, X_test_ideal, y_test_hr, meta_test = gather_subjects(test_subjects)

print("\nDataset shapes")
print("Train noisy:", X_train_noisy.shape)
print("Train ideal:", X_train_ideal.shape)
print("Val noisy:", X_val_noisy.shape)
print("Val ideal:", X_val_ideal.shape)
print("Test noisy:", X_test_noisy.shape)
print("Test ideal:", X_test_ideal.shape)

In [ ]:
# ============================================================
# 9. tf.data pipelines
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X_noisy, X_ideal, y_hr, batch_size=128, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X_noisy, X_ideal, y_hr))
    if training:
        ds = ds.shuffle(min(len(X_noisy), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_noisy, X_train_ideal, y_train_hr, batch_size=BATCH_SIZE, training=True)
val_ds = make_dataset(X_val_noisy, X_val_ideal, y_val_hr, batch_size=BATCH_SIZE, training=False)
test_ds = make_dataset(X_test_noisy, X_test_ideal, y_test_hr, batch_size=BATCH_SIZE, training=False)

In [ ]:
# ============================================================
# 10. Generator and Discriminator
# ============================================================
# Paper:
# Generator:
# - encoder-decoder
# - conv filters 64,128,256
# - kernel 16, stride 2
# - layer norm + leaky ReLU in encoder
# - decoder deconv filters 256,128,64
# - layer norm + ReLU in decoder
# - final tanh
#
# Discriminator:
# - conv filters 512,256,128,64
# - kernel 16, stride 2
# - leaky ReLU + batch norm
# - flatten + dense(1)
# ============================================================

def build_generator_with_hr_head(input_shape=(TRAIN_WINDOW_SAMPLES, 1)):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv1D(64, kernel_size=16, strides=2, padding="same")(inp)
    x = layers.LayerNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    skip1 = x

    x = layers.Conv1D(128, kernel_size=16, strides=2, padding="same")(x)
    x = layers.LayerNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    skip2 = x

    x = layers.Conv1D(256, kernel_size=16, strides=2, padding="same")(x)
    x = layers.LayerNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    bottleneck = x

    hr_head = layers.GlobalAveragePooling1D()(bottleneck)
    hr_head = layers.Dense(128, activation="relu")(hr_head)
    hr_out = layers.Dense(1, name="hr_output")(hr_head)

    x = layers.Conv1DTranspose(256, kernel_size=16, strides=2, padding="same")(bottleneck)
    x = layers.LayerNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Concatenate()([x, skip2])

    x = layers.Conv1DTranspose(128, kernel_size=16, strides=2, padding="same")(x)
    x = layers.LayerNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Concatenate()([x, skip1])

    x = layers.Conv1DTranspose(64, kernel_size=16, strides=2, padding="same")(x)
    x = layers.LayerNormalization()(x)
    x = layers.ReLU()(x)

    signal_out = layers.Conv1D(
        1, kernel_size=15, padding="same", activation="tanh", name="signal_output"
    )(x)

    model = models.Model(inp, [signal_out, hr_out], name="Generator")
    return model
    
def build_discriminator(input_shape=(TRAIN_WINDOW_SAMPLES, 1)):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv1D(512, kernel_size=16, strides=2, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv1D(256, kernel_size=16, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv1D(128, kernel_size=16, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv1D(64, kernel_size=16, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Flatten()(x)
    out = layers.Dense(1)(x)

    model = models.Model(inp, out, name="Discriminator")
    return model

generator=build_generator_with_hr_head()
discriminator = build_discriminator()

print("\nGenerator summary")
generator.summary()

print("\nDiscriminator summary")
discriminator.summary()

In [ ]:
# ============================================================
# 11. Losses and optimizers
# ============================================================
# Adversarial loss from GAN idea in paper.
# We also add L1 reconstruction loss to stabilize training.
# The paper emphasizes adversarial mapping to ideal representation.
# In practice, L1 helps much more on DALIA-only training.
# ============================================================

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

lr_schedule_g = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-4,
    decay_steps=10000,
    decay_rate=0.9,
    staircase=True
)

lr_schedule_d = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=5e-5,
    decay_steps=10000,
    decay_rate=0.9,
    staircase=True
)

gen_optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule_g, beta_1=0.5)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule_d, beta_1=0.5)

LAMBDA_L1 = 50.0
LAMBDA_HR = 1.0
mse_loss = tf.keras.losses.MeanSquaredError()

def discriminator_loss(real_logits, fake_logits):
    real_labels = tf.ones_like(real_logits) * 0.9
    fake_labels = tf.zeros_like(fake_logits)

    real_loss = bce(real_labels, real_logits)
    fake_loss = bce(fake_labels, fake_logits)
    return real_loss + fake_loss

def generator_loss(fake_logits, fake_signal, target_signal, pred_hr, true_hr):
    adv_loss = bce(tf.ones_like(fake_logits), fake_logits)
    l1_loss = tf.reduce_mean(tf.abs(fake_signal - target_signal))
    hr_loss = mse_loss(true_hr, pred_hr)
    total = adv_loss + LAMBDA_L1 * l1_loss + LAMBDA_HR * hr_loss
    return total, adv_loss, l1_loss, hr_loss

In [ ]:
# ============================================================
# 12. Training / validation steps
# ============================================================

@tf.function
def train_step(noisy_ppg, ideal_ppg, true_hr):
    true_hr = tf.cast(tf.reshape(true_hr, (-1, 1)), tf.float32)

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_ppg, pred_hr = generator(noisy_ppg, training=True)

        real_logits = discriminator(ideal_ppg, training=True)
        fake_logits = discriminator(fake_ppg, training=True)

        d_loss = discriminator_loss(real_logits, fake_logits)
        g_total, g_adv, g_l1, g_hr = generator_loss(
            fake_logits, fake_ppg, ideal_ppg, pred_hr, true_hr
        )

    gen_grads = gen_tape.gradient(g_total, generator.trainable_variables)
    disc_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return d_loss, g_total, g_adv, g_l1, g_hr

@tf.function
def val_step(noisy_ppg, ideal_ppg, true_hr):
    true_hr = tf.cast(tf.reshape(true_hr, (-1, 1)), tf.float32)

    fake_ppg, pred_hr = generator(noisy_ppg, training=False)

    real_logits = discriminator(ideal_ppg, training=False)
    fake_logits = discriminator(fake_ppg, training=False)

    d_loss = discriminator_loss(real_logits, fake_logits)
    g_total, g_adv, g_l1, g_hr = generator_loss(
        fake_logits, fake_ppg, ideal_ppg, pred_hr, true_hr
    )

    return d_loss, g_total, g_adv, g_l1, g_hr, fake_ppg, pred_hr


In [ ]:
# ============================================================
# 13. Helper: HR evaluation from generator outputs
# ============================================================

def predict_hr_batch(ppg_batch):
    hrs = []
    for i in range(len(ppg_batch)):
        hrs.append(estimate_hr_from_generated_ppg_robust(ppg_batch[i, :, 0], fs=TARGET_FS))
    return np.array(hrs, dtype=np.float32)

def regression_metrics(y_true, y_pred):
    mask = ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "corr": np.nan,
            "acc_pm3": np.nan,
            "n_valid": 0
        }

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    corr = pearsonr(y_true, y_pred)[0] if len(y_true) > 1 else np.nan
    acc_pm3 = np.mean(np.abs(y_true - y_pred) <= 3.0)

    return {
        "mae": mae,
        "rmse": rmse,
        "corr": corr,
        "acc_pm3": acc_pm3,
        "n_valid": len(y_true)
    }
def aggregate_hr_from_consecutive_windows(ppg_windows, fs=TARGET_FS, group_size=3):
    """
    Combine consecutive generated 4-second windows.
    group_size=3 -> 12 seconds
    """
    pred_hrs = []

    for i in range(0, len(ppg_windows) - group_size + 1):
        merged = []
        for j in range(group_size):
            merged.append(ppg_windows[i + j, :, 0])
        merged = np.concatenate(merged, axis=0)

        hr = estimate_hr_from_generated_ppg_robust(merged, fs=fs)
        pred_hrs.append(hr)

    return np.array(pred_hrs, dtype=np.float32)

def evaluate_subjectwise_long_window(generator, subject_list, group_size=3):
    """
    Evaluate using merged windows per subject.
    group_size=3 means 12-second evaluation.
    """
    y_true_all = []
    y_pred_all = []

    for s in subject_list:
        X_sub = subject_processed[s]["X_noisy"]
        y_sub = subject_processed[s]["y_hr"]

        fake_sub = generator.predict(X_sub, batch_size=BATCH_SIZE, verbose=0)

        pred_hr_sub = aggregate_hr_from_consecutive_windows(
            fake_sub, fs=TARGET_FS, group_size=group_size
        )

        true_hr_sub = []
        for i in range(0, len(y_sub) - group_size + 1):
            true_hr_sub.append(np.mean(y_sub[i:i+group_size]))
        true_hr_sub = np.array(true_hr_sub, dtype=np.float32)

        mask = ~np.isnan(pred_hr_sub)
        y_true_all.append(true_hr_sub[mask])
        y_pred_all.append(pred_hr_sub[mask])

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    return y_true_all, y_pred_all


def evaluate_subjectwise_long_window_with_hr_head(generator, subject_list, group_size=3):
    y_true_all = []
    y_pred_signal_all = []
    y_pred_head_all = []

    for s in subject_list:
        X_sub = subject_processed[s]["X_noisy"]
        y_sub = subject_processed[s]["y_hr"]

        fake_sub, hr_head_sub = generator.predict(X_sub, batch_size=BATCH_SIZE, verbose=0)
        hr_head_sub = hr_head_sub.reshape(-1)

        pred_hr_signal = aggregate_hr_from_consecutive_windows(
            fake_sub, fs=TARGET_FS, group_size=group_size
        )

        true_hr_sub = []
        pred_hr_head_agg = []

        for i in range(0, len(y_sub) - group_size + 1):
            true_hr_sub.append(np.mean(y_sub[i:i+group_size]))
            pred_hr_head_agg.append(np.mean(hr_head_sub[i:i+group_size]))

        true_hr_sub = np.array(true_hr_sub, dtype=np.float32)
        pred_hr_head_agg = np.array(pred_hr_head_agg, dtype=np.float32)

        mask_signal = ~np.isnan(pred_hr_signal)

        y_true_all.append(true_hr_sub[mask_signal])
        y_pred_signal_all.append(pred_hr_signal[mask_signal])
        y_pred_head_all.append(pred_hr_head_agg[mask_signal])

    y_true_all = np.concatenate(y_true_all)
    y_pred_signal_all = np.concatenate(y_pred_signal_all)
    y_pred_head_all = np.concatenate(y_pred_head_all)

    return y_true_all, y_pred_signal_all, y_pred_head_all

In [ ]:
# ============================================================
# 14. Full training loop with tqdm progress bar
# ============================================================
# Real-time display:
# - generator loss
# - discriminator loss
# - validation loss
# - validation MAE
# - validation ±3 BPM accuracy
#
# Best model is saved ONLY when overall validation MAE improves.
# Not saved every epoch unless it is globally best.
# ============================================================

history = {
    "train_g_total": [],
    "train_g_adv": [],
    "train_g_l1": [],
    "train_d": [],
    "val_g_total": [],
    "val_g_adv": [],
    "val_g_l1": [],
    "val_d": [],
    "val_mae": [],
    "val_rmse": [],
    "val_corr": [],
    "val_acc_pm3": []
}

best_val_mae = np.inf
best_epoch = -1

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    # ---------- TRAIN ----------
    train_d_losses = []
    train_g_losses = []
    train_g_adv_losses = []
    train_g_l1_losses = []

    pbar = tqdm(train_ds, total=len(list(train_ds)), desc=f"Train Epoch {epoch}", leave=False)

    for noisy_batch, ideal_batch, hr_batch in pbar:
    d_loss, g_total, g_adv, g_l1, g_hr = train_step(noisy_batch, ideal_batch, hr_batch)

        d_loss_v = float(d_loss.numpy())
        g_total_v = float(g_total.numpy())
        g_adv_v = float(g_adv.numpy())
        g_l1_v = float(g_l1.numpy())

        train_d_losses.append(d_loss_v)
        train_g_losses.append(g_total_v)
        train_g_adv_losses.append(g_adv_v)
        train_g_l1_losses.append(g_l1_v)
        train_g_hr_losses.append(float(g_hr.numpy()))

        pbar.set_postfix({
            "g_total": f"{np.mean(train_g_losses):.4f}",
            "g_adv": f"{np.mean(train_g_adv_losses):.4f}",
            "g_l1": f"{np.mean(train_g_l1_losses):.4f}",
            "g_hr": f"{np.mean(train_g_hr_losses):.4f}",
            "d_loss": f"{np.mean(train_d_losses):.4f}"
        })
    train_g_hr_losses = []
    # ---------- VALIDATION ----------
    val_d_losses = []
    val_g_losses = []
    val_g_adv_losses = []
    val_g_l1_losses = []

    all_val_preds = []
    all_val_hr_head_preds = []
    all_val_fake_ppg = []

    for noisy_batch, ideal_batch, hr_batch in val_ds:
        d_loss, g_total, g_adv, g_l1, g_hr, fake_ppg, pred_hr = val_step(noisy_batch, ideal_batch, hr_batch)

        val_d_losses.append(float(d_loss.numpy()))
        val_g_losses.append(float(g_total.numpy()))
        val_g_adv_losses.append(float(g_adv.numpy()))
        val_g_l1_losses.append(float(g_l1.numpy()))
        all_val_hr_head_preds.append(pred_hr.numpy().reshape(-1))
        all_val_fake_ppg.append(fake_ppg.numpy())
        
        preds_hr = predict_hr_batch(fake_ppg.numpy())
        all_val_preds.append(preds_hr)

    all_val_hr_head_preds = np.concatenate(all_val_hr_head_preds, axis=0)
    all_val_fake_ppg = np.concatenate(all_val_fake_ppg, axis=0)
    
    all_val_preds = np.concatenate(all_val_preds, axis=0)
    val_metrics = regression_metrics(y_val_hr, all_val_hr_head_preds)

    train_g_mean = np.mean(train_g_losses)
    train_g_adv_mean = np.mean(train_g_adv_losses)
    train_g_l1_mean = np.mean(train_g_l1_losses)
    train_d_mean = np.mean(train_d_losses)

    val_g_mean = np.mean(val_g_losses)
    val_g_adv_mean = np.mean(val_g_adv_losses)
    val_g_l1_mean = np.mean(val_g_l1_losses)
    val_d_mean = np.mean(val_d_losses)

    history["train_g_total"].append(train_g_mean)
    history["train_g_adv"].append(train_g_adv_mean)
    history["train_g_l1"].append(train_g_l1_mean)
    history["train_d"].append(train_d_mean)
    history["val_g_total"].append(val_g_mean)
    history["val_g_adv"].append(val_g_adv_mean)
    history["val_g_l1"].append(val_g_l1_mean)
    history["val_d"].append(val_d_mean)

    history["val_mae"].append(val_metrics["mae"])
    history["val_rmse"].append(val_metrics["rmse"])
    history["val_corr"].append(val_metrics["corr"])
    history["val_acc_pm3"].append(val_metrics["acc_pm3"])

    print(
        f"train_g={train_g_mean:.4f} | train_d={train_d_mean:.4f} | "
        f"val_g={val_g_mean:.4f} | val_d={val_d_mean:.4f} | "
        f"val_MAE={val_metrics['mae']:.4f} | val_RMSE={val_metrics['rmse']:.4f} | "
        f"val_corr={val_metrics['corr']:.4f} | val_acc(±3bpm)={val_metrics['acc_pm3']:.4f}"
    )

    # Save overall best only
    if val_metrics["mae"] < best_val_mae:
        best_val_mae = val_metrics["mae"]
        best_epoch = epoch

        generator.save_weights(BEST_GEN_PATH)
        discriminator.save_weights(BEST_DISC_PATH)

        print(f"Best model updated at epoch {epoch} | best val MAE = {best_val_mae:.4f}")

# Save last model after all epochs
generator.save_weights(LAST_GEN_PATH)
discriminator.save_weights(LAST_DISC_PATH)

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best generator saved to:", BEST_GEN_PATH)
print("Last generator saved to:", LAST_GEN_PATH)

In [ ]:
# ============================================================
# 15. Load best model before final test
# ============================================================
generator.load_weights(BEST_GEN_PATH)



In [ ]:
# ============================================================
# 16. Test evaluation
# ============================================================

y_true_final, y_pred_signal_final, y_pred_head_final = evaluate_subjectwise_long_window_with_hr_head(
    generator,
    test_subjects,
    group_size=3
)

metrics_signal = regression_metrics(y_true_final, y_pred_signal_final)
metrics_head = regression_metrics(y_true_final, y_pred_head_final)

print("\n========== TEST RESULTS ==========")
print("\n[From generated signal peaks]")
print(f"MAE:       {metrics_signal['mae']:.4f}")
print(f"RMSE:      {metrics_signal['rmse']:.4f}")
print(f"Corr:      {metrics_signal['corr']:.4f}")
print(f"Acc ±3:    {metrics_signal['acc_pm3']:.4f}")

print("\n[From HR head]")
print(f"MAE:       {metrics_head['mae']:.4f}")
print(f"RMSE:      {metrics_head['rmse']:.4f}")
print(f"Corr:      {metrics_head['corr']:.4f}")
print(f"Acc ±3:    {metrics_head['acc_pm3']:.4f}")

# A "test loss" from training perspective:
# use generator/discriminator validation-style losses on test set
test_d_losses, test_g_losses = [], []

for noisy_batch, ideal_batch, hr_batch in test_ds:
    d_loss, g_total, g_adv, g_l1, fake_ppg = val_step(noisy_batch, ideal_batch)
    test_d_losses.append(float(d_loss.numpy()))
    test_g_losses.append(float(g_total.numpy()))

print(f"Test Generator Loss:   {np.mean(test_g_losses):.4f}")
print(f"Test Discriminator Loss:{np.mean(test_d_losses):.4f}")


In [ ]:
# ============================================================
# 17. Training curves
# ============================================================

epochs_range = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history["train_g_total"], label="Train G loss")
plt.plot(epochs_range, history["val_g_total"], label="Val G loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Generator Loss Curves")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history["train_d"], label="Train D loss")
plt.plot(epochs_range, history["val_d"], label="Val D loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Discriminator Loss Curves")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history["val_mae"], label="Val MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE (BPM)")
plt.title("Validation MAE")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, history["val_acc_pm3"], label="Val Accuracy (±3 BPM)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy within ±3 BPM")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# ============================================================
# 19. Scatter plot of true vs predicted HR
# ============================================================

mask = ~np.isnan(y_test_pred_hr)
plt.figure(figsize=(6, 6))
plt.scatter(y_test_hr[mask], y_test_pred_hr[mask], alpha=0.4)
mn = min(np.min(y_test_hr[mask]), np.min(y_test_pred_hr[mask]))
mx = max(np.max(y_test_hr[mask]), np.max(y_test_pred_hr[mask]))
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel("True HR (BPM)")
plt.ylabel("Predicted HR (BPM)")
plt.title("True vs Predicted HR")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# 20. Show random 20 samples and say right/wrong
# ============================================================
# "Right" = prediction within ±3 BPM
# ============================================================

valid_indices = np.where(~np.isnan(y_test_pred_hr))[0]
sample_indices = np.random.choice(valid_indices, size=min(20, len(valid_indices)), replace=False)

print("\n========== RANDOM 20 TEST SAMPLES ==========\n")

for count, idx in enumerate(sample_indices, start=1):
    true_hr = y_test_hr[idx]
    pred_hr = y_test_pred_hr[idx]
    err = abs(true_hr - pred_hr)
    status = "RIGHT" if err <= 3.0 else "WRONG"
    subj, segid = meta_test[idx]

    print(
        f"{count:02d}. Subject={subj}, Segment={segid}, "
        f"True HR={true_hr:.2f}, Pred HR={pred_hr:.2f}, "
        f"Abs Err={err:.2f} -> {status}"
    )

In [ ]:
# ============================================================
# 21. Plot some qualitative examples
# ============================================================

num_examples = min(5, len(sample_indices))
plt.figure(figsize=(14, 3 * num_examples))

for row, idx in enumerate(sample_indices[:num_examples], start=1):
    noisy = X_test_noisy[idx, :, 0]
    fake = all_test_fake[idx, :, 0]
    ideal = X_test_ideal[idx, :, 0]

    t = np.arange(TRAIN_WINDOW_SAMPLES) / TARGET_FS

    plt.subplot(num_examples, 1, row)
    plt.plot(t, noisy, label="Noisy Input PPG", alpha=0.7)
    plt.plot(t, fake, label="Generated PPG", alpha=0.9)
    plt.plot(t, ideal, label="Ideal Target", alpha=0.7)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(f"Example {row}: True HR={y_test_hr[idx]:.2f}, Pred HR={y_test_pred_hr[idx]:.2f}")
    plt.grid(True)
    plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 22. Final summary print
# ============================================================

print("\n========== FINAL SUMMARY ==========")
print(f"Best epoch based on validation MAE: {best_epoch}")
print(f"Best model path: {BEST_GEN_PATH}")
print(f"Last model path: {LAST_GEN_PATH}")
print(f"Test MAE: {test_metrics['mae']:.4f} BPM")
print(f"Test RMSE: {test_metrics['rmse']:.4f} BPM")
print(f"Test correlation: {test_metrics['corr']:.4f}")
print(f"Test accuracy within ±3 BPM: {test_metrics['acc_pm3']:.4f}")